# Yelp — Baseline Linear Regression
**Features**: 6 LOO numerical + OHE(state) + OHE(city) + multi-hot(categories)
**Model**: Default sklearn LinearRegression
**Saves to**: `baseline_LR/`


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Load data
STEP 3 — Build feature matrix (numerical + OHE + multi-hot)
STEP 4 — Train & evaluate
STEP 5 — Feature importance
STEP 6 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics       import mean_squared_error, mean_absolute_error
from scipy.sparse          import csr_matrix, hstack
from sklearn.linear_model import LinearRegression

In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\baseline_LR"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print(f"Out dir: {OUT_DIR}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
NUM_COLS = [
    'user_avg_rating',    'user_rating_count',
    'business_avg_rating','business_rating_count', 'business_rating_std',
    'days_since_review'
]
TARGET      = 'stars'
CAT_STATE   = 'state'
CAT_CITY    = 'city'
CAT_CATS    = 'categories'


---
## Step 3 — Build Feature Matrix

**Why sparse matrix?**
OHE for 1,165 cities + multi-hot for 1,281 category tokens creates
a very wide but sparse matrix. scipy.sparse.hstack keeps it memory efficient.

**Design choices:**
- state: 23 unique values → OHE is clean and interpretable
- city: 1,165 unique values → OHE creates many columns but handle_unknown='ignore'
  silently zeros out unseen cities in val/test
- categories: multi-label → multi-hot (same as MovieLens genres)


In [ ]:
# ── OHE: state (23 unique — low cardinality, clean OHE) ──
ohe_state = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
state_tr = ohe_state.fit_transform(df_train[[CAT_STATE]].fillna('Unknown'))
state_v  = ohe_state.transform(df_val[[CAT_STATE]].fillna('Unknown'))
state_te = ohe_state.transform(df_test[[CAT_STATE]].fillna('Unknown'))

# ── OHE: city (1,165 unique — medium cardinality) ──
ohe_city = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
city_tr = ohe_city.fit_transform(df_train[[CAT_CITY]].fillna('Unknown'))
city_v  = ohe_city.transform(df_val[[CAT_CITY]].fillna('Unknown'))
city_te = ohe_city.transform(df_test[[CAT_CITY]].fillna('Unknown'))

# ── Multi-hot: categories (1,281 unique tokens) ──
cats_tr = df_train[CAT_CATS].fillna('Unknown').str.get_dummies('|')
cat_cols = cats_tr.columns
cats_v  = df_val[CAT_CATS].fillna('Unknown').str.get_dummies('|').reindex(columns=cat_cols, fill_value=0)
cats_te = df_test[CAT_CATS].fillna('Unknown').str.get_dummies('|').reindex(columns=cat_cols, fill_value=0)

# ── Scale numerical features ──
scaler  = StandardScaler()
num_tr  = scaler.fit_transform(df_train[NUM_COLS])
num_v   = scaler.transform(df_val[NUM_COLS])
num_te  = scaler.transform(df_test[NUM_COLS])

# ── Stack all into sparse matrix ──
X_train = hstack([csr_matrix(num_tr), state_tr, city_tr, csr_matrix(cats_tr.values)])
X_val   = hstack([csr_matrix(num_v),  state_v,  city_v,  csr_matrix(cats_v.values)])
X_test  = hstack([csr_matrix(num_te), state_te, city_te, csr_matrix(cats_te.values)])

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values

feat_names = (NUM_COLS
              + ohe_state.get_feature_names_out([CAT_STATE]).tolist()
              + ohe_city.get_feature_names_out([CAT_CITY]).tolist()
              + list(cat_cols))

print(f"Feature matrix — train: {X_train.shape}")
print(f"  {len(NUM_COLS)} numerical + {state_tr.shape[1]} state + "
      f"{city_tr.shape[1]} city + {cats_tr.shape[1]} categories")


---
## Step 4 — Train & Evaluate

In [ ]:
def get_metrics(y_true, y_pred):
    rmse = round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4)
    mae  = round(float(mean_absolute_error(y_true, y_pred)), 4)
    return rmse, mae


In [ ]:
t0    = time.perf_counter()
model = LinearRegression()
model.fit(X_train, y_train)
train_time = time.perf_counter() - t0

tr_rmse,  tr_mae  = get_metrics(y_train, model.predict(X_train))
val_rmse, val_mae = get_metrics(y_val,   model.predict(X_val))
te_rmse,  te_mae  = get_metrics(y_test,  model.predict(X_test))

print("=" * 50)
print("Linear Regression — Baseline Results")
print("=" * 50)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.2f}s")
print(f"  Train-Test gap : {round(te_rmse - tr_rmse, 4)}")


---
## Step 5 — Feature Importance
Top 15 features by absolute coefficient value.

In [ ]:
coefs  = model.coef_.ravel()
imp_df = (pd.DataFrame({'feature': feat_names, 'coef': coefs})
            .assign(abs_coef=lambda x: x['coef'].abs())
            .sort_values('abs_coef', ascending=False)
            .head(15))
print("Top 15 features by |coefficient|:")
print(imp_df[['feature','coef']].to_string(index=False))


---
## Step 6 — Save Results

In [ ]:
result = {
    'model'        : 'LinearRegression (Baseline)',
    'train_rmse'   : tr_rmse, 'val_rmse'  : val_rmse, 'test_rmse'  : te_rmse,
    'train_mae'    : tr_mae,  'val_mae'   : val_mae,  'test_mae'   : te_mae,
    'train_time_s' : round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(OUT_DIR, 'lr_results.csv'), index=False)
with open(os.path.join(OUT_DIR, 'lr_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved → {OUT_DIR}")
print(f"Test RMSE: {te_rmse}  Test MAE: {te_mae}")
